# Setup and Configuration

In [32]:
# Cell 1 — Import Libraries
import pandas as pd
import numpy as np
import sqlite3
import os
from pathlib import Path

print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")
print("All libraries imported successfully.")

Pandas version: 3.0.1
NumPy version: 2.4.4
All libraries imported successfully.


In [33]:
# Cell 2 — Configure Paths (updated for actual folder structure)
from pathlib import Path

PROJECT_ROOT = Path(r'C:\Users\samip\Desktop\Work\Projects\IPEDS Analytics project')

RAW_DATA  = PROJECT_ROOT / 'Data' / 'raw'
PROCESSED = PROJECT_ROOT / 'Data' / 'processed'
DB_PATH   = PROJECT_ROOT / 'Data' / 'ipeds.db'

# Updated to 2019-2024
YEARS = [2019, 2020, 2021, 2022, 2023, 2024]

MIDWEST_STATES = ['KS', 'MO', 'NE', 'OK', 'IA', 'CO']

PROCESSED.mkdir(parents=True, exist_ok=True)

print(f'Project root : {PROJECT_ROOT}')
print(f'Raw data     : {RAW_DATA}')
print(f'Years        : {YEARS}')
print(f'States       : {MIDWEST_STATES}')

Project root : C:\Users\samip\Desktop\Work\Projects\IPEDS Analytics project
Raw data     : C:\Users\samip\Desktop\Work\Projects\IPEDS Analytics project\Data\raw
Years        : [2019, 2020, 2021, 2022, 2023, 2024]
States       : ['KS', 'MO', 'NE', 'OK', 'IA', 'CO']


In [34]:
# Cell 3 — Verify all files are present (matches actual folder structure)
import os

def find_file(year, prefix):
    """
    Looks inside raw/{year}/{PREFIX}{year}/ folder
    Returns path to _rv version if exists, otherwise plain version
    """
    folder = RAW_DATA / str(year) / f'{prefix}{year}'
    rv_file  = folder / f'{prefix.lower()}{year}_rv.csv'
    std_file = folder / f'{prefix.lower()}{year}.csv'
    
    if rv_file.exists():
        return rv_file
    elif std_file.exists():
        return std_file
    else:
        return None

found   = []
missing = []

for year in YEARS:
    for prefix in ['HD', 'EF', 'GR']:
        # EF files have D suffix
        actual_prefix = f'EF{year}D' if prefix == 'EF' else f'{prefix}{year}'
        folder = RAW_DATA / str(year) / actual_prefix
        
        # Look for _rv first, then plain
        rv   = folder / f'{actual_prefix.lower()}_rv.csv'
        plain = folder / f'{actual_prefix.lower()}.csv'
        
        if rv.exists():
            found.append((year, prefix, str(rv)))
        elif plain.exists():
            found.append((year, prefix, str(plain)))
        else:
            missing.append(f'{year} / {prefix} — looked in {folder}')

print(f'Files found  : {len(found)} / 18')
print(f'Files missing: {len(missing)}')

if missing:
    print('\nMISSING:')
    for m in missing:
        print(f'  {m}')
else:
    print('\nAll files confirmed. Ready to proceed.')

print('\nFound files:')
for year, prefix, path in found:
    print(f'  {year} {prefix}: {Path(path).name}')

Files found  : 18 / 18
Files missing: 0

All files confirmed. Ready to proceed.

Found files:
  2019 HD: hd2019.csv
  2019 EF: ef2019d_rv.csv
  2019 GR: gr2019_rv.csv
  2020 HD: hd2020.csv
  2020 EF: ef2020d_rv.csv
  2020 GR: gr2020_rv.csv
  2021 HD: hd2021.csv
  2021 EF: ef2021d_rv.csv
  2021 GR: gr2021_rv.csv
  2022 HD: hd2022.csv
  2022 EF: ef2022d_rv.csv
  2022 GR: gr2022_rv.csv
  2023 HD: hd2023.csv
  2023 EF: ef2023d_rv.csv
  2023 GR: gr2023_rv.csv
  2024 HD: hd2024.csv
  2024 EF: ef2024d.csv
  2024 GR: gr2024.csv


# Load Raw Data

In [35]:
# Cell 4 — Load HD files (fixed for BOM encoding in 2023/2024)
HD_COLS = [
    'UNITID',   # Unique institution ID — join key
    'INSTNM',   # Institution name
    'STABBR',   # State abbreviation
    'CONTROL',  # 1=Public, 2=Private nonprofit, 3=For-profit
    'ICLEVEL',  # 1=Four-year, 2=Two-year
    'LOCALE',   # Urbanization code
    'INSTSIZE', # Size category
    'HBCU',     # HBCU flag
    'LANDGRNT', # Land grant flag
]

hd_frames = []

for year in YEARS:
    folder = RAW_DATA / str(year) / f'HD{year}'
    rv     = folder / f'hd{year}_rv.csv'
    plain  = folder / f'hd{year}.csv'
    fpath  = rv if rv.exists() else plain

    # utf-8-sig handles BOM characters in 2023/2024 files
    # latin-1 handles special characters in older files
    # We try utf-8-sig first, fall back to latin-1
    try:
        df = pd.read_csv(fpath, encoding='utf-8-sig', low_memory=False)
    except:
        df = pd.read_csv(fpath, encoding='latin-1', low_memory=False)

    # Rename BOM-affected column if present
    df.columns = df.columns.str.replace('ï»¿', '', regex=False).str.strip()

    # Keep only columns we need
    available = [c for c in HD_COLS if c in df.columns]
    df = df[available].copy()
    df['year'] = year
    hd_frames.append(df)
    print(f'HD{year}: {len(df):,} rows | columns: {df.columns.tolist()}')

hd_all = pd.concat(hd_frames, ignore_index=True)

# Filter: Public (CONTROL=1) + 4-year (ICLEVEL=1) + Midwest states
hd_filtered = hd_all[
    (hd_all['CONTROL'] == 1) &
    (hd_all['ICLEVEL'] == 1) &
    (hd_all['STABBR'].isin(MIDWEST_STATES))
].copy()

print(f'\nTotal before filter : {len(hd_all):,} rows')
print(f'Total after filter  : {len(hd_filtered):,} rows')
print(f'Unique institutions : {hd_filtered["UNITID"].nunique()}')
print(f'\nInstitutions per state:')
print(hd_filtered.groupby('STABBR')['UNITID'].nunique().sort_values(ascending=False))

HD2019: 6,559 rows | columns: ['UNITID', 'INSTNM', 'STABBR', 'CONTROL', 'ICLEVEL', 'LOCALE', 'INSTSIZE', 'HBCU', 'LANDGRNT', 'year']
HD2020: 6,440 rows | columns: ['UNITID', 'INSTNM', 'STABBR', 'CONTROL', 'ICLEVEL', 'LOCALE', 'INSTSIZE', 'HBCU', 'LANDGRNT', 'year']
HD2021: 6,289 rows | columns: ['UNITID', 'INSTNM', 'STABBR', 'CONTROL', 'ICLEVEL', 'LOCALE', 'INSTSIZE', 'HBCU', 'LANDGRNT', 'year']
HD2022: 6,256 rows | columns: ['UNITID', 'INSTNM', 'STABBR', 'CONTROL', 'ICLEVEL', 'LOCALE', 'INSTSIZE', 'HBCU', 'LANDGRNT', 'year']
HD2023: 6,163 rows | columns: ['UNITID', 'INSTNM', 'STABBR', 'CONTROL', 'ICLEVEL', 'LOCALE', 'INSTSIZE', 'HBCU', 'LANDGRNT', 'year']
HD2024: 6,072 rows | columns: ['UNITID', 'INSTNM', 'STABBR', 'CONTROL', 'ICLEVEL', 'LOCALE', 'INSTSIZE', 'HBCU', 'LANDGRNT', 'year']

Total before filter : 37,779 rows
Total after filter  : 468 rows
Unique institutions : 90

Institutions per state:
STABBR
CO    27
OK    19
MO    17
NE    11
IA     8
KS     8
Name: UNITID, dtype: int6

In [36]:
# Cell 6 — Add readable labels to HD data
locale_map = {
    11: 'City - Large',    12: 'City - Midsize',   13: 'City - Small',
    21: 'Suburb - Large',  22: 'Suburb - Midsize',  23: 'Suburb - Small',
    31: 'Town - Fringe',   32: 'Town - Distant',   33: 'Town - Remote',
    41: 'Rural - Fringe',  42: 'Rural - Distant',  43: 'Rural - Remote'
}
locale_simple = {
    11: 'City',   12: 'City',   13: 'City',
    21: 'Suburb', 22: 'Suburb', 23: 'Suburb',
    31: 'Town',   32: 'Town',   33: 'Town',
    41: 'Rural',  42: 'Rural',  43: 'Rural'
}
size_map = {
    1: 'Under 1,000',
    2: '1,000 - 4,999',
    3: '5,000 - 9,999',
    4: '10,000 - 19,999',
    5: '20,000 and above'
}

hd_filtered = hd_filtered.copy()
hd_filtered['locale_label']  = hd_filtered['LOCALE'].map(locale_map)
hd_filtered['locale_simple'] = hd_filtered['LOCALE'].map(locale_simple)
hd_filtered['size_label']    = hd_filtered['INSTSIZE'].map(size_map)
hd_filtered['is_hbcu']       = hd_filtered['HBCU'].map({1: 'Yes', 2: 'No'})
hd_filtered['is_landgrant']  = hd_filtered['LANDGRNT'].map({1: 'Yes', 2: 'No'})

print('Labels added successfully.')
print(f'\nLocale distribution (2024):')
print(hd_filtered[hd_filtered['year']==2024]['locale_simple'].value_counts())
print(f'\nSize distribution (2024):')
print(hd_filtered[hd_filtered['year']==2024]['size_label'].value_counts())
print(f'\nSample rows:')
print(hd_filtered[hd_filtered['year']==2024][
    ['INSTNM','STABBR','locale_simple','size_label','is_hbcu','is_landgrant']
].head(5).to_string(index=False))

Labels added successfully.

Locale distribution (2024):
locale_simple
City      40
Town      24
Suburb    10
Rural      8
Name: count, dtype: int64

Size distribution (2024):
size_label
1,000 - 4,999       29
5,000 - 9,999       20
10,000 - 19,999     14
20,000 and above    13
Under 1,000          2
Name: count, dtype: int64

Sample rows:
                                               INSTNM STABBR locale_simple       size_label is_hbcu is_landgrant
                               Adams State University     CO          Town    1,000 - 4,999      No           No
                               Aims Community College     CO          City    5,000 - 9,999      No           No
                           Arapahoe Community College     CO        Suburb  10,000 - 19,999      No           No
University of Colorado Denver/Anschutz Medical Campus     CO          City 20,000 and above      No           No
              University of Colorado Colorado Springs     CO          City  10,000 - 19,999   

In [37]:
# Cell 6 — Load EF D files (retention rate + student-faculty ratio)
EF_COLS = ['UNITID', 'RET_PCF', 'STUFACR', 'UGENTERN']

ef_frames = []

for year in YEARS:
    folder = RAW_DATA / str(year) / f'EF{year}D'
    rv     = folder / f'ef{year}d_rv.csv'
    plain  = folder / f'ef{year}d.csv'
    fpath  = rv if rv.exists() else plain

    try:
        df = pd.read_csv(fpath, encoding='utf-8-sig', low_memory=False)
    except:
        df = pd.read_csv(fpath, encoding='latin-1', low_memory=False)

    # Fix BOM in column names
    df.columns = df.columns.str.replace('ï»¿', '', regex=False).str.strip()

    # Keep only columns we need
    available = [c for c in EF_COLS if c in df.columns]
    df = df[available].copy()
    df['year'] = year

    # Replace negative values (IPEDS suppression codes) with NaN
    for col in ['RET_PCF', 'STUFACR', 'UGENTERN']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
            df.loc[df[col] < 0, col] = np.nan

    ef_frames.append(df)
    valid = df['RET_PCF'].notna().sum()
    print(f'EF{year}D: {len(df):,} rows | {valid:,} with valid retention rate')

ef_all = pd.concat(ef_frames, ignore_index=True)
print(f'\nTotal EF rows: {len(ef_all):,}')
print(f'\nRetention rate sample stats:')
print(ef_all['RET_PCF'].describe().round(1))

EF2019D: 5,885 rows | 5,188 with valid retention rate
EF2020D: 5,845 rows | 5,187 with valid retention rate
EF2021D: 5,771 rows | 5,100 with valid retention rate
EF2022D: 5,706 rows | 5,068 with valid retention rate
EF2023D: 5,646 rows | 5,007 with valid retention rate
EF2024D: 5,578 rows | 4,947 with valid retention rate

Total EF rows: 34,431

Retention rate sample stats:
count    30497.0
mean        71.4
std         18.5
min          0.0
25%         61.0
50%         73.0
75%         84.0
max        100.0
Name: RET_PCF, dtype: float64


In [38]:
# Cell 7 — Load GR files (corrected GRTYPE values)
GR_COLS = ['UNITID', 'GRTYPE', 'CHRTSTAT', 'GRTOTLT']

gr_frames = []

for year in YEARS:
    folder = RAW_DATA / str(year) / f'GR{year}'
    rv     = folder / f'gr{year}_rv.csv'
    plain  = folder / f'gr{year}.csv'
    fpath  = rv if rv.exists() else plain

    try:
        df = pd.read_csv(fpath, encoding='utf-8-sig', low_memory=False)
    except:
        df = pd.read_csv(fpath, encoding='latin-1', low_memory=False)

    df.columns = df.columns.str.replace('ï»¿', '', regex=False).str.strip()
    available = [c for c in GR_COLS if c in df.columns]
    df = df[available].copy()

    # Clean GRTOTLT
    df['GRTOTLT'] = pd.to_numeric(df['GRTOTLT'], errors='coerce')
    df.loc[df['GRTOTLT'] < 0, 'GRTOTLT'] = np.nan

    # Cohort size — GRTYPE=2, CHRTSTAT=12
    cohort = (
        df[(df['GRTYPE'] == 2) & (df['CHRTSTAT'] == 12)]
        [['UNITID', 'GRTOTLT']]
        .rename(columns={'GRTOTLT': 'cohort_size'})
    )

    # Completers within 150% time — GRTYPE=3, CHRTSTAT=13
    completers = (
        df[(df['GRTYPE'] == 3) & (df['CHRTSTAT'] == 13)]
        [['UNITID', 'GRTOTLT']]
        .rename(columns={'GRTOTLT': 'completers_150'})
    )

    # Merge and calculate graduation rate
    gr_year = cohort.merge(completers, on='UNITID', how='inner')
    gr_year['grad_rate_6yr'] = (
        gr_year['completers_150'] / gr_year['cohort_size'] * 100
    ).round(1)

    # Cap at 100%
    gr_year.loc[gr_year['grad_rate_6yr'] > 100, 'grad_rate_6yr'] = np.nan
    gr_year['year'] = year

    gr_frames.append(gr_year)
    valid = gr_year['grad_rate_6yr'].notna().sum()
    print(f'GR{year}: {len(gr_year):,} institutions | {valid:,} with valid grad rate | avg: {gr_year["grad_rate_6yr"].mean():.1f}%')

gr_all = pd.concat(gr_frames, ignore_index=True)
print(f'\nTotal GR rows: {len(gr_all):,}')
print(f'\nGrad rate stats (all years):')
print(gr_all['grad_rate_6yr'].describe().round(1))

GR2019: 2,253 institutions | 2,253 with valid grad rate | avg: 52.2%
GR2020: 2,231 institutions | 2,231 with valid grad rate | avg: 52.7%
GR2021: 2,214 institutions | 2,214 with valid grad rate | avg: 53.4%
GR2022: 2,242 institutions | 2,242 with valid grad rate | avg: 53.2%
GR2023: 2,228 institutions | 2,228 with valid grad rate | avg: 53.1%
GR2024: 2,243 institutions | 2,243 with valid grad rate | avg: 53.0%

Total GR rows: 13,411

Grad rate stats (all years):
count    13411.0
mean        52.9
std         20.8
min          1.1
25%         38.0
50%         53.0
75%         67.4
max        100.0
Name: grad_rate_6yr, dtype: float64


# Load Build Analytical Dataset

In [45]:
# Cell 8 — Merge HD + EF + GR into one analytical dataset
# Start with HD (our filtered Midwest public 4-year institutions)
merged = hd_filtered.copy()
print(f'Base (HD filtered)  : {len(merged):,} rows | {merged["UNITID"].nunique()} institutions')

# Join enrollment + retention data
merged = merged.merge(
    ef_all[['UNITID', 'year', 'RET_PCF', 'STUFACR', 'UGENTERN']],
    on=['UNITID', 'year'],
    how='left'
)
print(f'After EF join       : {len(merged):,} rows')

# Join graduation rates
merged = merged.merge(
    gr_all[['UNITID', 'year', 'grad_rate_6yr', 'cohort_size', 'completers_150']],
    on=['UNITID', 'year'],
    how='left'
)
print(f'After GR join       : {len(merged):,} rows')

# Rename columns to readable names
merged = merged.rename(columns={
    'UNITID'    : 'unitid',
    'INSTNM'    : 'institution_name',
    'STABBR'    : 'state',
    'CONTROL'   : 'control',
    'ICLEVEL'   : 'level',
    'LOCALE'    : 'locale_code',
    'INSTSIZE'  : 'size_code',
    'HBCU'      : 'hbcu_flag',
    'LANDGRNT'  : 'landgrant_flag',
    'RET_PCF'   : 'retention_rate',
    'STUFACR'   : 'stufac_ratio',
    'UGENTERN'  : 'entering_students'
})

print(f'\nFinal dataset       : {len(merged):,} rows | {merged["unitid"].nunique()} institutions')
print(f'Columns             : {merged.columns.tolist()}')
print(f'\nSample rows:')
print(merged[['institution_name', 'state', 'year', 'retention_rate', 
              'grad_rate_6yr', 'stufac_ratio']].head(8).to_string())

Base (HD filtered)  : 468 rows | 90 institutions
After EF join       : 468 rows
After GR join       : 468 rows

Final dataset       : 468 rows | 90 institutions
Columns             : ['unitid', 'institution_name', 'state', 'control', 'level', 'locale_code', 'size_code', 'hbcu_flag', 'landgrant_flag', 'year', 'locale_label', 'locale_simple', 'size_label', 'is_hbcu', 'is_landgrant', 'retention_rate', 'stufac_ratio', 'entering_students', 'grad_rate_6yr', 'cohort_size', 'completers_150']

Sample rows:
                                        institution_name state  year  retention_rate  grad_rate_6yr  stufac_ratio
0                                 Adams State University    CO  2019            59.0           29.9          15.0
1                             Arapahoe Community College    CO  2019             NaN           25.1          23.0
2  University of Colorado Denver/Anschutz Medical Campus    CO  2019            70.0           51.8          17.0
3                University of Colorado C

In [46]:
# Cell 9 — Data quality check
key_cols = ['retention_rate', 'grad_rate_6yr', 'stufac_ratio', 'entering_students']

print('=== Missing Value Summary ===')
for col in key_cols:
    total   = len(merged)
    missing = merged[col].isna().sum()
    pct     = missing / total * 100
    print(f'{col:<22}: {missing:>3} missing ({pct:.1f}%)')

print('\n=== Key Variable Distributions ===')
print(merged[key_cols].describe().round(1))

print('\n=== Rows per Year ===')
print(merged.groupby('year')['unitid'].count())

print('\n=== Avg Grad Rate by State (2024) ===')
print(
    merged[merged['year'] == 2024]
    .groupby('state')[['retention_rate', 'grad_rate_6yr']]
    .mean()
    .round(1)
    .sort_values('grad_rate_6yr', ascending=False)
)

=== Missing Value Summary ===
retention_rate        : 112 missing (23.9%)
grad_rate_6yr         :  56 missing (12.0%)
stufac_ratio          :  38 missing (8.1%)
entering_students     :  50 missing (10.7%)

=== Key Variable Distributions ===
       retention_rate  grad_rate_6yr  stufac_ratio  entering_students
count           356.0          412.0         430.0              418.0
mean             71.7           45.6          16.6             2572.0
std              13.6           18.0           3.8             2210.9
min               0.0           11.1           2.0              193.0
25%              63.8           31.5          15.0             1053.2
50%              72.0           44.3          17.0             1718.0
75%              82.0           57.0          19.0             3350.5
max             100.0           99.9          28.0             9334.0

=== Rows per Year ===
year
2019    82
2020    74
2021    75
2022    76
2023    79
2024    82
Name: unitid, dtype: int64

=== Avg

In [47]:
# Cell 10 — Add derived features for regression and dashboard
# Sort for correct year-over-year calculation
merged = merged.sort_values(['unitid', 'year']).reset_index(drop=True)

# Year-over-year enrollment change (%)
merged['enroll_yoy_pct'] = (
    merged.groupby('unitid')['entering_students']
    .pct_change() * 100
).round(1)

# Cap extreme outliers — likely data issues
merged.loc[merged['enroll_yoy_pct'].abs() > 50, 'enroll_yoy_pct'] = np.nan

# Enrollment size bucket
merged['enroll_bucket'] = pd.cut(
    merged['entering_students'],
    bins=[0, 500, 1500, 3000, 6000, 999999],
    labels=['Very Small (<500)', 'Small (500-1,500)',
            'Medium (1,500-3,000)', 'Large (3,000-6,000)', 'Very Large (6,000+)']
)

# COVID period flag
merged['covid_period'] = merged['year'].isin([2020, 2021]).map(
    {True: 'COVID Year', False: 'Normal Year'}
)

# Retention rate category
merged['retention_category'] = pd.cut(
    merged['retention_rate'],
    bins=[0, 60, 70, 80, 90, 100],
    labels=['Below 60%', '60-70%', '70-80%', '80-90%', 'Above 90%']
)

# Grad rate category
merged['grad_rate_category'] = pd.cut(
    merged['grad_rate_6yr'],
    bins=[0, 30, 45, 60, 75, 100],
    labels=['Below 30%', '30-45%', '45-60%', '60-75%', 'Above 75%']
)

# Flag institutions with consistent enrollment decline
# (more than 2 years of negative YoY growth)
decline_count = (
    merged[merged['enroll_yoy_pct'] < 0]
    .groupby('unitid')['year']
    .count()
    .rename('decline_years')
)
merged = merged.merge(decline_count, on='unitid', how='left')
merged['decline_years'] = merged['decline_years'].fillna(0).astype(int)
merged['enrollment_trend'] = merged['decline_years'].apply(
    lambda x: 'Consistent Decline' if x >= 3 else 'Stable/Growing'
)

print('Derived features added successfully.')
print(f'\nEnrollment buckets (2024):')
print(merged[merged['year']==2024]['enroll_bucket'].value_counts())
print(f'\nRetention categories (2024):')
print(merged[merged['year']==2024]['retention_category'].value_counts())
print(f'\nGrad rate categories (2024):')
print(merged[merged['year']==2024]['grad_rate_category'].value_counts())
print(f'\nEnrollment trend (2024):')
print(merged[merged['year']==2024]['enrollment_trend'].value_counts())

Derived features added successfully.

Enrollment buckets (2024):
enroll_bucket
Small (500-1,500)       23
Medium (1,500-3,000)    22
Large (3,000-6,000)     13
Very Large (6,000+)     10
Very Small (<500)        7
Name: count, dtype: int64

Retention categories (2024):
retention_category
60-70%       19
70-80%       18
80-90%       12
Below 60%     7
Above 90%     5
Name: count, dtype: int64

Grad rate categories (2024):
grad_rate_category
30-45%       24
45-60%       20
Below 30%    15
60-75%       11
Above 75%     4
Name: count, dtype: int64

Enrollment trend (2024):
enrollment_trend
Stable/Growing        49
Consistent Decline    33
Name: count, dtype: int64


# Export

In [48]:
# Cell 11 — Export to CSV and SQLite

# ── Export to CSV ────────────────────────────────────────────
csv_path = PROCESSED / 'ipeds_midwest_clean.csv'
merged.to_csv(csv_path, index=False)
print(f'CSV exported: {csv_path}')
print(f'Shape: {merged.shape[0]:,} rows x {merged.shape[1]} columns')

# ── Export to SQLite ─────────────────────────────────────────
conn = sqlite3.connect(DB_PATH)

# Table 1: Full panel dataset — used for all SQL EDA queries
merged.to_sql('ipeds_main', conn, if_exists='replace', index=False)
print(f'\nTable ipeds_main     : {len(merged):,} rows')

# Table 2: Institution reference — one row per institution (2024 snapshot)
inst_cols = [
    'unitid', 'institution_name', 'state',
    'locale_simple', 'size_label',
    'is_hbcu', 'is_landgrant', 'enrollment_trend'
]
institutions = (
    merged[merged['year'] == 2024][inst_cols]
    .drop_duplicates()
    .reset_index(drop=True)
)
institutions.to_sql('institutions', conn, if_exists='replace', index=False)
print(f'Table institutions   : {len(institutions):,} rows')

# Table 3: State summary — pre-aggregated for Power BI map visual
state_summary = (
    merged.groupby(['state', 'year'])
    .agg(
        institutions        = ('unitid', 'nunique'),
        avg_retention       = ('retention_rate', 'mean'),
        avg_grad_rate       = ('grad_rate_6yr', 'mean'),
        avg_stufac          = ('stufac_ratio', 'mean'),
        total_enrollment    = ('entering_students', 'sum')
    )
    .round(1)
    .reset_index()
)
state_summary.to_sql('state_summary', conn, if_exists='replace', index=False)
print(f'Table state_summary  : {len(state_summary):,} rows')

# ── Verification query ───────────────────────────────────────
print('\n=== Verification — State Summary 2024 ===')
verify = pd.read_sql('''
    SELECT state,
           institutions,
           avg_retention,
           avg_grad_rate,
           total_enrollment
    FROM state_summary
    WHERE year = 2024
    ORDER BY avg_grad_rate DESC
''', conn)
print(verify.to_string(index=False))

conn.close()
print(f'\nDatabase saved: {DB_PATH}')

CSV exported: C:\Users\samip\Desktop\Work\Projects\IPEDS Analytics project\Data\processed\ipeds_midwest_clean.csv
Shape: 468 rows x 28 columns

Table ipeds_main     : 468 rows
Table institutions   : 82 rows
Table state_summary  : 36 rows

=== Verification — State Summary 2024 ===
state  institutions  avg_retention  avg_grad_rate  total_enrollment
   IA             3           86.7           72.6           16257.0
   KS             8           77.1           52.8           24116.0
   MO            16           75.7           48.4           40877.0
   NE             9           73.8           47.7           13146.0
   CO            27           69.9           44.1           73067.0
   OK            19           66.9           35.1           34388.0

Database saved: C:\Users\samip\Desktop\Work\Projects\IPEDS Analytics project\Data\ipeds.db


---
## ETL Complete

| | |
|---|---|
| Institutions | 90 |
| Years | 2019 – 2024 |
| States | KS, MO, NE, OK, IA, CO |
| Total rows | 468 |
| Output CSV | Data/processed/ipeds_midwest_clean.csv |
| Output DB | Data/ipeds.db |

**Next:** `02_eda.ipynb` — SQL queries + Python visualizations